# Setup

## Import

In [23]:
import json
import logging
from pathlib import Path
from torch.utils.data import DataLoader
from datasets import load_from_disk
from model_testing.model import build_model_and_transforms, get_device, CollateFn
from model_testing.utils import run_inference, compute_metrics

## Costanti

In [24]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
# Constants
IMAGENET_1K_DATASET_PATH = Path("../preprocessed_datasets/imagenet_1k_preprocessed")
IMAGENET_R_DATASET_PATH = Path("../preprocessed_datasets/imagenet_r_preprocessed")

RESULTS_BASELINE_PATH = Path("../baseline_metrics.json")
RESULTS_DRIFT_PATH = Path("../drift_metrics.json")

BATCH_SIZE = 64
NUM_WORKERS = 4
TOP_K = 5

# Test

## Setup modello e dataset

### Load device

In [25]:
device = get_device()
logger.info(f"Utilizing device: {device}")

INFO:__main__:Utilizing device: cuda


### Load dataset

In [26]:
if not IMAGENET_1K_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed dataset not found in {IMAGENET_1K_DATASET_PATH}. "
        "Please, execute the preprocessing script first."
    )
if not IMAGENET_R_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed dataset not found in {IMAGENET_R_DATASET_PATH}. "
        "Please, execute the preprocessing script first."
    )

logger.info("Loading preprocessed dataset...")
dataset_base = load_from_disk(str(IMAGENET_1K_DATASET_PATH))
dataset_r = load_from_disk(str(IMAGENET_R_DATASET_PATH))


INFO:__main__:Loading preprocessed dataset...


### Load model

In [27]:
logger.info("Loading ResNet50...")
model, preprocess = build_model_and_transforms()
model.to(device)

dataloader_base = DataLoader(
    dataset_base,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=CollateFn(preprocess=preprocess),
)
dataloader_r = DataLoader(
    dataset_r,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=CollateFn(preprocess=preprocess),
)


INFO:__main__:Loading ResNet50...


## Inferenza

In [ ]:
y_true_base, y_pred_base, topk_correct_base, features_base = run_inference(
    model, dataloader_base, device, k=TOP_K, return_features=True
)
y_true_r, y_pred_r, topk_correct_r, features_r = run_inference(
    model, dataloader_r, device, k=TOP_K, return_features=True)

## Calcolo metriche

In [ ]:
metrics_base = compute_metrics(y_true_base, y_pred_base, topk_correct_base)
metrics_r = compute_metrics(y_true_r, y_pred_r, topk_correct_r)

### Stampa e salvataggio metriche

In [ ]:
from model_testing.utils import save_features_csv

logger.info("Baseline metrics computed:")
for key, value in metrics_base.items():
    logger.info(f"  {key}: {value}")

with open(RESULTS_BASELINE_PATH, "w") as f:
    json.dump(metrics_base, f, indent=2)
logger.info(f"Metrics saved in {RESULTS_BASELINE_PATH}")
save_features_csv(features_base, y_true_base, y_pred_base, "../baseline_features.csv")

In [ ]:
logger.info("Drift metrics computed:")
for key, value in metrics_r.items():
    logger.info(f"  {key}: {value}")
with open(RESULTS_DRIFT_PATH, "w") as f:
    json.dump(metrics_r, f, indent=2)
logger.info(f"Metrics saved in {RESULTS_DRIFT_PATH}")
save_features_csv(features_r, y_true_r, y_pred_r, "../drift_features.csv")
